In [1]:
import tensorflow
from tensorflow.keras.models import Sequential
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import Embedding, LSTM, Dense, Bidirectional
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

I0000 00:00:1782277259.273160    1767 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1782277259.610102    1767 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1782277262.597863    1767 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [3]:
# importando o dataframe
df = pd.read_csv('taylor_swift_lyrics.csv', encoding='latin-1')
df.head()

,artist,album,track_title,track_n,lyric,line,year
0,Taylor Swift,Taylor Swift,Tim McGraw,1,He said the way my blue eyes shined,1,2006
1,Taylor Swift,Taylor Swift,Tim McGraw,1,Put those Georgia stars to shame that night,2,2006
2,Taylor Swift,Taylor Swift,Tim McGraw,1,"I said, ""That's a lie""",3,2006
3,Taylor Swift,Taylor Swift,Tim McGraw,1,Just a boy in a Chevy truck,4,2006
4,Taylor Swift,Taylor Swift,Tim McGraw,1,That had a tendency of gettin' stuck,5,2006


In [4]:
# lista os valores que interessam, ou seja, as lyrics
lyrics = list(df['lyric'].str.lower().values) 
lyrics[:10]

['he said the way my blue eyes shined',
 'put those georgia stars to shame that night',
 'i said, "that\'s a lie"',
 'just a boy in a chevy truck',
 "that had a tendency of gettin' stuck",
 'on backroads at night',
 'and i was right there beside him all summer long',
 'and then the time we woke up to find that summer gone',
 'but when you think tim mcgraw',
 'i hope you think my favorite song']

In [5]:
print(lyrics[2])

i said, "that's a lie"


In [6]:
# instanciando o tokenizer
tk = Tokenizer()

In [7]:
# mapeando as lyrcs 
tk.fit_on_texts(lyrics)

# quantidade total de palavras no tokenizer
total_words = len(tk.word_index) + 1

print(tk.word_index, total_words)

{'you': 1, 'i': 2, 'the': 3, 'and': 4, 'to': 5, 'me': 6, 'it': 7, 'a': 8, 'in': 9, 'my': 10, 'we': 11, 'your': 12, 'of': 13, 'all': 14, 'but': 15, 'that': 16, 'is': 17, 'know': 18, 'like': 19, "i'm": 20, 'on': 21, 'this': 22, 'oh': 23, 'be': 24, "don't": 25, 'so': 26, 'was': 27, "you're": 28, 'now': 29, "it's": 30, 'back': 31, 'when': 32, 'what': 33, 'just': 34, 'never': 35, 'are': 36, 'for': 37, 'with': 38, 'time': 39, 'up': 40, 'out': 41, 'love': 42, 'at': 43, 'do': 44, 'got': 45, 'if': 46, "'cause": 47, 'say': 48, 'one': 49, 'see': 50, 'baby': 51, 'want': 52, 'think': 53, 'go': 54, 'down': 55, 'ever': 56, 'they': 57, 'said': 58, 'been': 59, 'not': 60, 'he': 61, 'had': 62, "can't": 63, 'could': 64, "i'll": 65, 'look': 66, 'were': 67, 'no': 68, 'can': 69, 'how': 70, 'would': 71, 'come': 72, 'stay': 73, 'have': 74, 'take': 75, 'off': 76, 'made': 77, 'why': 78, 'wanna': 79, 'there': 80, 'yeah': 81, 'shake': 82, 'here': 83, 'night': 84, 'ey': 85, 'way': 86, 'tell': 87, 'about': 88, 'wish

In [8]:
# lista que guarda as sequencias
input_sequences = []
for line in lyrics:
    # transforma texto em número
    token_list = tk.texts_to_sequences([line])[0]
    # adiciona as versoes da linha
    for i in range(1, len(token_list)):
        n_gram_sequence = token_list[:i+1]
        input_sequences.append(n_gram_sequence)

input_sequences[:10]

[[61, 58],
 [61, 58, 3],
 [61, 58, 3, 86],
 [61, 58, 3, 86, 10],
 [61, 58, 3, 86, 10, 318],
 [61, 58, 3, 86, 10, 318, 115],
 [61, 58, 3, 86, 10, 318, 115, 604],
 [194, 263],
 [194, 263, 1117],
 [194, 263, 1117, 605]]

In [9]:
# tamanho máximo das sequências
max_sequence_len = max([len(x) for x in input_sequences])
# padding das sequencias para formar a matriz
input_sequences = np.array(pad_sequences(input_sequences, maxlen=max_sequence_len, padding='pre'))
input_sequences[10]

array([   0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,  194,  263, 1117,  605,    5], dtype=int32)

In [10]:
# definindo os valores de X e os de Y
xs, labels = input_sequences[:, :-1], input_sequences[:, -1]
# transformando os valores de Y em um vetor onehot
ys = tensorflow.keras.utils.to_categorical(labels, num_classes=total_words)

xs[10], ys[10]

(array([   0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,  194,  263, 1117,  605], dtype=int32),
 array([0., 0., 0., ..., 0., 0., 0.], shape=(2408,)))

In [11]:
# criando a rede neural
model = Sequential()

# camada de embedding com 100 dimensões
model.add(Embedding(total_words, output_dim=100, input_length=max_sequence_len-1))
# camada que guarda a memória
model.add(Bidirectional(LSTM(150)))
# camadas densas
model.add(Dense(total_words, activation='relu'))
model.add(Dense(total_words//2, activation='relu'))
model.add(Dense(total_words//2//2, activation='relu'))
model.add(Dense(total_words, activation='softmax'))

/home/liperasz/miniconda3/envs/card-25/lib/python3.11/site-packages/keras/src/layers/core/embedding.py:103: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(
E0000 00:00:1782277288.422330    1767 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [12]:
# definindo o optimizador
adam = Adam(learning_rate=0.001)
# compilando
model.compile(loss='categorical_crossentropy', optimizer=adam, metrics=['accuracy'])

In [13]:
# definindo callbacks
# para se ficar 5 épocas sem melhorar a val_loss
early_stopping = EarlyStopping(
    monitor='val_loss', 
    patience=5, 
    restore_best_weights=True,
    verbose=1
)

# reduz o lr 20% do valor se não melhorar em 3 epocas
reduce_lr = ReduceLROnPlateau(
    monitor='val_loss', 
    factor=0.2, 
    patience=3, 
    min_lr=0.00001,
    verbose=1
)

# salva o melhor modelo localmente
model_checkpoint = ModelCheckpoint(
    filepath='model.h5',
    monitor='val_loss', 
    save_best_only=True,
    verbose=1
)

# lista de callbacks
callbacks_list = [early_stopping, reduce_lr, model_checkpoint]

In [ ]:
# treinamento
history = model.fit(
    xs,
    ys,
    epochs=100,
    verbose=1,
    batch_size=64,
    callbacks=callbacks_list
)

Epoch 1/100


W0000 00:00:1782269616.440874    1819 cpu_allocator_impl.cc:82] Allocation of 294768096 exceeds 10% of free system memory.


479/479 ━━━━━━━━━━━━━━━━━━━━ 0s 162ms/step - accuracy: 0.0520 - loss: 5.7165

/home/liperasz/miniconda3/envs/card-25/lib/python3.11/site-packages/keras/src/callbacks/early_stopping.py:99: UserWarning: Early stopping conditioned on metric `val_loss` which is not available. Available metrics are: accuracy,loss
  current = self.get_monitor_value(logs)
/home/liperasz/miniconda3/envs/card-25/lib/python3.11/site-packages/keras/src/callbacks/callback_list.py:171: UserWarning: Learning rate reduction is conditioned on metric `val_loss` which is not available. Available metrics are: accuracy,loss,learning_rate.
  callback.on_epoch_end(epoch, logs)
/home/liperasz/miniconda3/envs/card-25/lib/python3.11/site-packages/keras/src/callbacks/model_checkpoint.py:329: UserWarning: Can save best model only with val_loss available.
  if self._should_save_model(epoch, batch, logs, filepath):



Epoch 1: finished saving model to model.h5
479/479 ━━━━━━━━━━━━━━━━━━━━ 84s 163ms/step - accuracy: 0.0533 - loss: 5.6929 - learning_rate: 0.0010
Epoch 2/100
479/479 ━━━━━━━━━━━━━━━━━━━━ 0s 154ms/step - accuracy: 0.0592 - loss: 5.4767

...


Epoch 96: finished saving model to model.h5
479/479 ━━━━━━━━━━━━━━━━━━━━ 66s 137ms/step - accuracy: 0.7874 - loss: 0.7924 - learning_rate: 0.0010
Epoch 97/100
479/479 ━━━━━━━━━━━━━━━━━━━━ 0s 136ms/step - accuracy: 0.7918 - loss: 0.7765


Epoch 97: finished saving model to model.h5
479/479 ━━━━━━━━━━━━━━━━━━━━ 66s 137ms/step - accuracy: 0.7842 - loss: 0.8025 - learning_rate: 0.0010
Epoch 98/100
479/479 ━━━━━━━━━━━━━━━━━━━━ 0s 137ms/step - accuracy: 0.7857 - loss: 0.7993


Epoch 98: finished saving model to model.h5
479/479 ━━━━━━━━━━━━━━━━━━━━ 66s 137ms/step - accuracy: 0.7849 - loss: 0.7969 - learning_rate: 0.0010
Epoch 99/100
479/479 ━━━━━━━━━━━━━━━━━━━━ 0s 129ms/step - accuracy: 0.7997 - loss: 0.7389


Epoch 99: finished saving model to model.h5
479/479 ━━━━━━━━━━━━━━━━━━━━ 62s 129ms/step - accuracy: 0.7929 - loss: 0.7620 - learning_rate: 0.0010
Epoch 100/100
479/479 ━━━━━━━━━━━━━━━━━━━━ 0s 137ms/step - accuracy: 0.7997 - loss: 0.7447


Epoch 100: finished saving model to model.h5
479/479 ━━━━━━━━━━━━━━━━━━━━ 66s 137ms/step - accuracy: 0.7933 - loss: 0.7680 - learning_rate: 0.0010


In [14]:
# carregando o modelo a partir do arquivo
model = tensorflow.keras.models.load_model('model.h5')

In [15]:
# função para plotar os graficos com historico do treinamento
def plot_graphs(history, string):
    plt.plot(history.history[string])
    plt.xlabel('Epochs')
    plt.ylabel(string)
    plt.show()

In [18]:
# deu erro na hora de plotar pq eu fechei o arquivo depois do treinamento, então os dados foram perdidos
plot_graphs(history, 'accuracy')
plot_graphs(history, 'loss')

NameError: name 'history' is not defined

In [19]:
# função para gerar letras de musicas
def generate_lyrics(seed_text, next_words, model, tokenizer, max_sequence_len):
    for _ in range(next_words):
        # converte texto inicial em tokens e aplica padding
        token_list = tokenizer.texts_to_sequences([seed_text])[0]
        token_list = pad_sequences([token_list], maxlen=max_sequence_len - 1, padding='pre')
        
        # preve a proxima palavra
        predicted = np.argmax(model.predict(token_list, verbose=0), axis=-1)
        
        # busca a proxima palavra
        output_word = tokenizer.index_word.get(predicted[0], "")
        
        if not output_word:
            break
            
        # adiciona a palavra ao texto
        seed_text += ' ' + output_word
        
    return seed_text

In [ ]:
# texto incicial
seed_text = "i've got a bad feeling about this"
# número máximo de palavras
n = 100

# gerando uma musica
music = generate_lyrics(
    seed_text=seed_text,
    next_words=n,
    model=model,
    tokenizer=tk,
    max_sequence_len=max_sequence_len
)

print(music)

i've got a bad feeling about this you smile you need to feel with me when i was feel down my love to me in me out in my beautiful my bad my smile to you down all just go back down my tragic my pretend time his coat sirens in me the my love me down one shotgun sin out of you in my my my bad cool the lucky one my love is my movie shake it off it made me crazy lovers one else to down my single hands in my truck the it off it down my love me down my clean yeah
